# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

**下のセル1を開いて、3つを設定してください。**

| 設定項目 | 説明 |
|---|---|
| `GEMINI_API_KEY` | **初回だけ**入力。2回目からは空欄でOK |
| `PEXELS_API_KEY` | **初回だけ**入力。2回目からは空欄でOK |
| `THEME` | **毎回**テーマを書き換える |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: ここだけ変える（毎回）                        ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキー（初回だけ入力。2回目以降は空欄のままでOK） ──
GEMINI_API_KEY = ''   # ← 初回だけ: 'AIza....' を貼り付ける
PEXELS_API_KEY = ''   # ← 初回だけ: 'xxxxx...' を貼り付ける

# ── テーマ（毎回変える） ─────────────────────────────────
THEME = 'ダイエット'   # ← ここにテーマを書く
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── 動画の長さ ───────────────────────────────────────────
DURATION = 45          # ← 秒数（30〜60）

# ── 画像スタイル ─────────────────────────────────────────
IMAGE_STYLE = 'realistic'   # 写真そのまま
# IMAGE_STYLE = 'anime'       # アニメ風
# IMAGE_STYLE = 'manga'       # 漫画風（白黒）
# IMAGE_STYLE = 'illustration' # イラスト風

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

print('設定内容を確認します...')
print(f'  テーマ      : {THEME}')
print(f'  動画の長さ  : {DURATION}秒')
print(f'  画像スタイル: {IMAGE_STYLE}')

# APIキーをドライブに保存＆読み込み
from google.colab import drive
drive.mount('/content/drive')

import os, json
CONFIG_PATH = '/content/drive/MyDrive/YouTube_Production/.config.json'
os.makedirs('/content/drive/MyDrive/YouTube_Production', exist_ok=True)

# 保存済みキーを読み込む
saved = {}
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        saved = json.load(f)

# 入力があれば上書き、なければ保存済みを使用
if GEMINI_API_KEY.strip():
    saved['GEMINI_API_KEY'] = GEMINI_API_KEY.strip()
if PEXELS_API_KEY.strip():
    saved['PEXELS_API_KEY'] = PEXELS_API_KEY.strip()

GEMINI_API_KEY = saved.get('GEMINI_API_KEY', '')
PEXELS_API_KEY = saved.get('PEXELS_API_KEY', '')

if not GEMINI_API_KEY:
    raise ValueError('❌ GEMINI_API_KEY が設定されていません。セル1の GEMINI_API_KEY = に貼り付けてください。')
if not PEXELS_API_KEY:
    raise ValueError('❌ PEXELS_API_KEY が設定されていません。セル1の PEXELS_API_KEY = に貼り付けてください。')

# ドライブに保存
with open(CONFIG_PATH, 'w') as f:
    json.dump(saved, f)

print()
print('✅ 設定完了！「ランタイム → すべてのセルを実行」で動画生成が始まります。')

In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg
print('✅ ツールのインストール完了')

In [ ]:
# 【自動】① YouTubeトレンド分析 → ② 台本生成（触らなくてOK）

import requests as _req
import json, re

GEMINI_URL = 'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent'

def call_gemini(prompt):
    res = _req.post(
        f'{GEMINI_URL}?key={GEMINI_API_KEY}',
        json={'contents': [{'parts': [{'text': prompt}]}],
              'generationConfig': {'temperature': 0.8, 'maxOutputTokens': 4096}},
        timeout=120
    )
    if res.status_code != 200:
        raise RuntimeError(f'Gemini APIエラー ({res.status_code}): {res.text[:300]}')
    return res.json()['candidates'][0]['content']['parts'][0]['text']

SCENE_COUNT = 4 if DURATION <= 35 else 5 if DURATION <= 50 else 6

print(f'🔍 「{THEME}」のYouTubeトレンドを分析中...')
trend = call_gemini(f'あなたはYouTubeショート動画のトレンドアナリストです。テーマ「{THEME}」（{DURATION}秒）について、バズりやすいフック・構成・差別化のコツを台本制作に直接使える形で簡潔にまとめてください。')
print('  ✓ トレンド分析完了')

print(f'📝 「{THEME}」の台本を生成中...')
raw_script = call_gemini(f"""
あなたはYouTubeショート動画の台本専門ライターです。
【テーマ】{THEME}【目標尺】{DURATION}秒（{SCENE_COUNT}シーン）【トレンド分析】{trend}

以下のフォーマットで出力してください。
[台本]
シーン1（フック）:（セリフ）
シーン2（問題提起）:（セリフ）
シーン3（本題①）:（セリフ）
シーン4（本題②）:（セリフ）
{'シーン5（まとめ）:（セリフ）' if SCENE_COUNT >= 5 else ''}{'シーン6（CTA）:（セリフ）' if SCENE_COUNT >= 6 else ''}
[キーワード]
scene1:（英語1〜3語）
scene2:（英語1〜3語）
scene3:（英語1〜3語）
scene4:（英語1〜3語）
{'scene5:（英語1〜3語）' if SCENE_COUNT >= 5 else ''}{'scene6:（英語1〜3語）' if SCENE_COUNT >= 6 else ''}
[秒数配分]
scene1:（秒数）
scene2:（秒数）
scene3:（秒数）
scene4:（秒数）
{'scene5:（秒数）' if SCENE_COUNT >= 5 else ''}{'scene6:（秒数）' if SCENE_COUNT >= 6 else ''}
""")

script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw_script) or type('x',(),({
    'group':lambda s,n:raw_script}))()).group(1)
kw_block   = re.search(r'\[キーワード\]([\s\S]*?)(?=\[秒数配分\])', raw_script)
time_block = re.search(r'\[秒数配分\]([\s\S]*?)$', raw_script)

keywords = []
if kw_block:
    for line in kw_block.group(1).split('\n'):
        m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
        if m: keywords.append(m.group(1).strip())
while len(keywords) < SCENE_COUNT: keywords.append('motivation')

timings = []
if time_block:
    for line in time_block.group(1).split('\n'):
        m = re.match(r'scene\d+[:\s]+(\d+)', line.strip(), re.I)
        if m: timings.append(int(m.group(1)))
while len(timings) < SCENE_COUNT: timings.append(DURATION // SCENE_COUNT)

scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
scene_texts = [s.strip() for s in scene_lines[1:] if s.strip()]
while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)

print(f'  ✓ 台本生成完了（{SCENE_COUNT}シーン）')
print('\n━'*20)
print(script_text[:500])
print('━'*20)

In [ ]:
# 【自動】③ 台本に合った画像を取得（触らなくてOK）

import time
from pathlib import Path
from datetime import datetime

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
safe_theme  = re.sub(r'[\\/:*?"<>|]', '_', THEME)[:25]
PROJECT_DIR = Path(f'/content/drive/MyDrive/YouTube_Production/{ts}_{safe_theme}')
IMG_DIR     = PROJECT_DIR / 'images'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)
(PROJECT_DIR / '台本.txt').write_text(script_text, encoding='utf-8')

print('🖼 台本に合った画像を取得中...')

def search_pexels(kw):
    r = _req.get('https://api.pexels.com/v1/search',
        headers={'Authorization': PEXELS_API_KEY},
        params={'query': kw, 'per_page': 1, 'orientation': 'portrait'}, timeout=15)
    if r.status_code != 200: return None
    p = r.json().get('photos', [])
    return p[0] if p else None

scenes = []
for i, (kw, dur) in enumerate(zip(keywords[:SCENE_COUNT], timings[:SCENE_COUNT])):
    time.sleep(0.6)
    fn  = f'scene_{i+1:02d}.jpg'
    pth = IMG_DIR / fn
    photo = search_pexels(kw)
    if photo:
        url = photo['src'].get('portrait') or photo['src'].get('large')
        r   = _req.get(url, timeout=30)
        if r.status_code == 200:
            pth.write_bytes(r.content)
            print(f'  ✓ シーン{i+1}: 「{kw}」')
        else:
            print(f'  ⚠ シーン{i+1}: DL失敗')
    else:
        print(f'  ⚠ シーン{i+1}: 画像見つからず')
    scenes.append({'scene': i+1, 'image': f'images/{fn}', 'keyword': kw,
                   'duration': dur, 'effect': 'zoom_in' if i%2==0 else 'zoom_out',
                   'text': scene_texts[i] if i < len(scene_texts) else ''})

(PROJECT_DIR / 'production_config.json').write_text(
    json.dumps({'theme': THEME, 'target_duration': DURATION, 'scene_count': SCENE_COUNT,
                'scenes': scenes, 'image_style': IMAGE_STYLE,
                'created_at': datetime.now().isoformat()}, ensure_ascii=False, indent=2), encoding='utf-8')
print('\n✅ 画像の取得が完了しました')

In [ ]:
# 【自動】④ 画像スタイル変換（触らなくてOK）

import cv2, numpy as np

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

if IMAGE_STYLE == 'realistic':
    print('🖼 スタイル: 写真そのまま（変換なし）')
else:
    print(f'🎨 「{IMAGE_STYLE}」スタイルに変換中...')
    fn = STYLES[IMAGE_STYLE]
    for s in scenes:
        p = PROJECT_DIR / s['image']
        if not p.exists(): continue
        img = cv2.imread(str(p))
        if img is not None:
            cv2.imwrite(str(p), fn(img))
            print(f"  ✓ シーン{s['scene']}")
print('✅ 画像スタイル処理完了')

In [ ]:
# 【自動】⑤ ナレーション音声を生成（触らなくてOK）

import subprocess, tempfile
from gtts import gTTS

TMP = Path(tempfile.mkdtemp(prefix='yt_'))

def clean(t):
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)','',t)
    t = re.sub(r'シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    return re.sub(r'\s+',' ',t).strip()

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],capture_output=True,text=True)
    if r.returncode!=0: raise RuntimeError(r.stderr[-500:])

print('🎙 ナレーション音声を生成中...')
wavs = []
for i,s in enumerate(scenes):
    t = clean(s.get('text',THEME)) or THEME
    mp3,wav = TMP/f'v{i}.mp3', TMP/f'v{i}.wav'
    try:
        gTTS(text=t,lang='ja').save(str(mp3))
        ff('-i',mp3,'-ar','44100','-ac','1',wav)
        wavs.append(wav)
        print(f'  ✓ シーン{i+1}')
    except Exception as e:
        print(f'  ⚠ シーン{i+1}: {e}')

VOICE = None
if wavs:
    lf = TMP/'vl.txt'
    lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
    comb,vaac = TMP/'vc.wav', TMP/'v.aac'
    ff('-f','concat','-safe','0','-i',lf,'-c','copy',comb)
    ff('-i',comb,'-c:a','aac','-ar','44100',vaac)
    VOICE = vaac
    print('\n✅ 音声生成完了')

In [ ]:
# 【自動】⑥ 動画を生成して完成（触らなくてOK）

W,H,FPS = 1080,1920,30

def clip(img,dur,out,eff):
    fr = int(dur*FPS); st = 0.15/max(fr,1)
    ze = f"min(1+{st:.6f}*on,1.15)" if eff=='zoom_in' else f"if(eq(on,1),1.15,max(1.0,zoom-{st:.6f}))"
    zp = f"zoompan=z='{ze}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={fr}:s={W}x{H}:fps={FPS}"
    sp = f"scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H}"
    ff('-loop','1','-i',img,'-vf',f'{sp},{zp}','-t',str(dur),'-an','-c:v','libx264','-preset','fast','-crf','22','-pix_fmt','yuv420p',out)

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

print('🎬 動画を生成中...')
clips=[]
for s in scenes:
    img=PROJECT_DIR/s['image']; out=TMP/f"c{s['scene']:02d}.mp4"
    if img.exists(): clip(img,s['duration'],out,s['effect']); print(f"  ✓ シーン{s['scene']}")
    else: ff('-f','lavfi','-i',f'color=black:s={W}x{H}:r={FPS}','-t',str(s['duration']),'-c:v','libx264','-preset','fast',out)
    clips.append(out)

lf=TMP/'cl.txt'; lf.write_text('\n'.join(f"file '{p}'" for p in clips))
mg=TMP/'m.mp4'; ff('-f','concat','-safe','0','-i',lf,'-c','copy',mg)

ah=(f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
    f"[V4+ Styles]\nFormat:Name,Fontname,Fontsize,PrimaryColour,OutlineColour,Bold,Outline,Shadow,Alignment,MarginV\n"
    f"Style:Default,Arial,72,&H00FFFFFF,&H00000000,-1,4,1,2,{int(H*0.18)}\n\n"
    f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n")
ev=[]; t=0.0
for s in scenes:
    ev.append(f"Dialogue:0,{at(t)},{at(t+s['duration'])},Default,,0,0,0,,{s['keyword'][:20]}")
    t+=s['duration']
sub=TMP/'s.ass'; sub.write_text(ah+'\n'.join(ev),encoding='utf-8')
se=str(sub).replace('\\','/').replace(':','\\:')

od=PROJECT_DIR/'output'; od.mkdir(exist_ok=True)
OUT=od/f'shorts_{datetime.now().strftime("%Y%m%d_%H%M%S")}.mp4'

if VOICE and VOICE.exists():
    ff('-i',mg,'-i',VOICE,'-vf',f'ass={se}','-map','0:v','-map','1:a',
       '-c:v','libx264','-preset','medium','-crf','20','-c:a','aac','-b:a','192k',
       '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),OUT)
else:
    ff('-i',mg,'-vf',f'ass={se}','-c:v','libx264','-preset','medium','-crf','20',
       '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),OUT)

mb=OUT.stat().st_size/1_048_576
print(f'\n🎉 動画完成！')
print(f'  テーマ  : {THEME}')
print(f'  サイズ  : {mb:.1f} MB')
print(f'  保存先  : マイドライブ → YouTube_Production → {PROJECT_DIR.name} → output')

In [ ]:
# 完成動画を確認する
from IPython.display import Video, display
import shutil
shutil.copy(str(OUT), '/content/preview.mp4')
print(f'▶ テーマ: {THEME} / スタイル: {IMAGE_STYLE}')
display(Video('/content/preview.mp4', width=360))